In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
# ============================================================
# Optimized 6-region boundaries + 2011-2021 cargo accident visualization
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature
from matplotlib.lines import Line2D

plt.rcParams.update({'font.sans-serif': ['Arial', 'DejaVu Sans'],
                     'axes.unicode_minus': False})

ACC   = rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet'
YEARS = (2011, 2021)

REGIONS = {
    'NW_Europe': {
        'Iberia_Atlantic':   [(36, 45, -15, -6)],
        'Biscay_Celtic':     [(45, 63, -15, -6), (43, 63, -6, 0)],
        'North_Sea':         [(48, 63, 0, 13)],
    },
    'NW_Atlantic': {
        'GoM_Caribbean': [(18, 31, -98, -60)],
        'NE_Coast':      [(31, 42, -82, -60), (42, 50, -67, -60)],
        'Newfoundland':  [(45, 55, -60, -46)],
    },
    'Asia': {
        'SE_Asia': [(-10, 20, 95, 130)],
        'E_Asia':  [(20, 46, 117, 146)],
    },
    'Med_Black': {
        'Med_West': [(30, 43, -6, 0), (30, 46, 0, 19)],
        'Med_East': [(30, 46, 19, 26.5), (30, 41, 26.5, 37)],
        'Black':    [(41, 47.5, 26.5, 42)],
    },
    'Baltic': {
        'Baltic': [(53, 66, 13, 30)],
    },
    'Indian_Ocean_N': {
        'Arabian': [(5, 30, 50, 78)],
        'BoB':     [(5, 26, 78, 95)],
    },
}

SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300']
COL    = dict(zip(REGIONS, SERIES))
GREY, LAND, OCEAN, INK, INK2 = '#b3b2ad', '#eceae4', '#f7f9fa', '#0b0b0b', '#52514e'
LABEL_POS = {
    'NW_Europe':      (-26, 70, 'left',  'bottom'),
    'NW_Atlantic':    (-99, 57, 'left',  'bottom'),
    'Asia':           (148, 47, 'left',  'bottom'),
    'Med_Black':      ( -7, 29, 'right', 'top'),
    'Baltic':         ( 32, 62, 'left',  'center'),
    'Indian_Ocean_N': ( 49,  3, 'right', 'top'),
}

FLAT = {(r, s): b for r, subs in REGIONS.items() for s, b in subs.items()}

def assign(lat, lon):
    for (r, s), boxes in FLAT.items():
        for (S, N, W, E) in boxes:
            if S <= lat < N and W <= lon < E:
                return r, s
    return None, None

acc = pd.read_parquet(ACC).dropna(subset=['lat', 'lon'])
acc = acc[acc.year.between(*YEARS)].copy()
lab = [assign(a, o) for a, o in zip(acc.lat, acc.lon)]
acc['region'], acc['subregion'] = [x[0] for x in lab], [x[1] for x in lab]

reg_n, sub_n = acc.region.value_counts(), acc.groupby(['region', 'subregion']).size()
n_in, n_all  = int(reg_n.sum()), len(acc)
print(f'{YEARS[0]}-{YEARS[1]}: total {n_all}, in regions {n_in} ({n_in/n_all*100:.1f}%)')
print(sub_n.sort_values(ascending=False).to_string())

def basemap(ax):
    ax.add_feature(cfeature.OCEAN, facecolor=OCEAN, zorder=0)
    ax.add_feature(cfeature.LAND,  facecolor=LAND,  zorder=0)
    ax.add_feature(cfeature.COASTLINE, lw=.35, edgecolor='#9a9992', zorder=1)

def boxes_of(r): return [b for s in REGIONS[r] for b in REGIONS[r][s]]
def env(bs):     return (min(b[0] for b in bs), max(b[1] for b in bs),
                         min(b[2] for b in bs), max(b[3] for b in bs))

# ---------- Fig 1 global overview ----------
fig = plt.figure(figsize=(17, 7.4))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -45, 80], crs=ccrs.PlateCarree()); basemap(ax)
o = acc[acc.region.isna()]
ax.scatter(o.lon, o.lat, s=3.5, c=GREY, alpha=.55, lw=0,
           transform=ccrs.PlateCarree(), zorder=2)
for r in REGIONS:
    g = acc[acc.region == r]
    ax.scatter(g.lon, g.lat, s=7, c=COL[r], alpha=.7, lw=0,
               transform=ccrs.PlateCarree(), zorder=3)
    for (S, N, W, E) in boxes_of(r):
        ax.add_patch(plt.Rectangle((W, S), E-W, N-S, transform=ccrs.PlateCarree(),
                                   fill=False, edgecolor=COL[r], lw=1.4, zorder=4))
    lx, ly, ha, va = LABEL_POS[r]
    ax.text(lx, ly, f'{r}  {reg_n[r]:,}', color=INK, fontsize=9.5, fontweight='bold',
            ha=ha, va=va, transform=ccrs.PlateCarree(), zorder=6,
            bbox=dict(boxstyle='round,pad=.25', fc='white', ec=COL[r], lw=1, alpha=.92))
gl = ax.gridlines(draw_labels=True, lw=.3, alpha=.25, color='#9a9992')
gl.top_labels = gl.right_labels = False
gl.xlabel_style = gl.ylabel_style = {'size': 7.5, 'color': INK2}
ax.legend(handles=[Line2D([], [], marker='o', ls='', ms=6, mfc=COL[r], mec='none', label=r)
                   for r in REGIONS] +
                  [Line2D([], [], marker='o', ls='', ms=5, mfc=GREY, mec='none',
                          label=f'outside {n_all-n_in:,}')],
          loc='lower left', fontsize=8, frameon=True, framealpha=.9)
ax.set_title(f'Cargo accidents {YEARS[0]}–{YEARS[1]} and study-region boundaries   '
             f'in regions {n_in:,} / total {n_all:,} ({n_in/n_all*100:.0f}%)',
             fontsize=12.5, color=INK, pad=10)
plt.tight_layout(); plt.show()

# ---------- Fig 2 accidents per sub-region ----------
d = sub_n.reset_index(name='n').sort_values('n')
fig, ax = plt.subplots(figsize=(9.5, 6.2))
y = np.arange(len(d))
ax.barh(y, d.n, height=.62, color=[COL[r] for r in d.region], zorder=3)
for yi, v in zip(y, d.n):
    ax.text(v + max(d.n)*.012, yi, f'{v:,}', va='center', fontsize=8.5, color=INK)
ax.set_yticks(y); ax.set_yticklabels(d.subregion, fontsize=9, color=INK)
ax.set_xlim(0, max(d.n)*1.12); ax.set_xlabel('Accidents', fontsize=9, color=INK2)
ax.grid(axis='x', lw=.4, alpha=.3, zorder=0); ax.set_axisbelow(True)
for sp in ('top', 'right', 'left'): ax.spines[sp].set_visible(False)
ax.spines['bottom'].set_color('#d5d4cf'); ax.tick_params(length=0, colors=INK2)
ax.legend(handles=[Line2D([], [], marker='s', ls='', ms=7, mfc=COL[r], mec='none',
                          label=f'{r}  {reg_n[r]:,}') for r in REGIONS],
          loc='lower right', fontsize=8.5, frameon=False)
ax.set_title(f'Cargo accidents per sub-region · {YEARS[0]}–{YEARS[1]}', fontsize=12, color=INK, pad=8)
plt.tight_layout(); plt.show()

# ---------- Fig 3 zoom on 6 regions to check edges ----------
fig, axes = plt.subplots(2, 3, figsize=(18, 10.5),
                         subplot_kw={'projection': ccrs.PlateCarree()})
for ax, r in zip(axes.ravel(), REGIONS):
    S, N, W, E = env(boxes_of(r)); mg = max(3, .12*max(N-S, E-W))
    ax.set_extent([W-mg, E+mg, max(-89, S-mg), min(89, N+mg)], crs=ccrs.PlateCarree())
    basemap(ax)
    sub = acc[(acc.lon > W-mg) & (acc.lon < E+mg) & (acc.lat > S-mg) & (acc.lat < N+mg)]
    ot = sub[sub.region != r]
    ax.scatter(ot.lon, ot.lat, s=5, c=GREY, alpha=.5, lw=0,
               transform=ccrs.PlateCarree(), zorder=2)
    g = sub[sub.region == r]
    ax.scatter(g.lon, g.lat, s=9, c=COL[r], alpha=.65, lw=0,
               transform=ccrs.PlateCarree(), zorder=3)
    for s in REGIONS[r]:
        for (bs_, bn, bw, be) in REGIONS[r][s]:
            ax.add_patch(plt.Rectangle((bw, bs_), be-bw, bn-bs_, transform=ccrs.PlateCarree(),
                                       fill=False, edgecolor=COL[r], lw=1.6, zorder=4))
        ss, nn, ww, ee = env(REGIONS[r][s])
        ax.text((ww+ee)/2, ss+.4, f'{s} {sub_n[(r, s)]:,}', fontsize=7.5, color=INK,
                ha='center', va='bottom', transform=ccrs.PlateCarree(), zorder=6,
                bbox=dict(boxstyle='round,pad=.2', fc='white', ec='none', alpha=.8))
    gl = ax.gridlines(draw_labels=True, lw=.3, alpha=.25, color='#9a9992')
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {'size': 6.5, 'color': INK2}
    ax.set_title(f'{r} · {reg_n[r]:,}', fontsize=10.5, color=COL[r], fontweight='bold')
plt.suptitle(f'Edge check per region · colored = in-region accidents · grey = outside accidents · {YEARS[0]}–{YEARS[1]}',
             fontsize=12, color=INK, y=.995)
plt.tight_layout(); plt.show()


In [ ]:
import itertools
allb = [(f'{r}/{s}', b) for r, subs in REGIONS.items() for s, bs in subs.items() for b in bs]
bad = [(k1, b1, k2, b2) for (k1, b1), (k2, b2) in itertools.combinations(allb, 2)
       if min(b1[1], b2[1]) > max(b1[0], b2[0]) and min(b1[3], b2[3]) > max(b1[2], b2[2])]
print('Overlapping boxes:', bad if bad else 'none')

m = pd.read_csv(rf'{DATA_ROOT}/climate_AIS_data/GMTDS/cargo_ship/organized/_manifest.csv')
m.columns = [c.strip().lstrip('\ufeff') for c in m.columns]
def in_gm(la, lo): return bool(((m.S <= la) & (m.N >= la) & (m.W <= lo) & (m.E >= lo)).any())
s = acc[acc.region.notna()].copy()
s['gm'] = [in_gm(a, o) for a, o in zip(s.lat, s.lon)]
print(f'\nAccidents without GMTDS traffic: {(~s.gm).sum()} / {len(s)} ({(~s.gm).mean()*100:.1f}%)')
print(s.groupby('subregion').gm.apply(lambda x: round((~x).mean()*100, 1))
       .sort_values(ascending=False).to_string())


In [ ]:
# ============================================================
# GMTDS coverage check: sub-region x box x month x actual raster values
# ============================================================
import os, glob, numpy as np, pandas as pd, rasterio

GMTDS    = rf'{DATA_ROOT}/climate_AIS_data/GMTDS/cargo_ship/organized'
GRID     = 2.0          # consistent with the M2 panel
DECIMATE = 8            # raster decimation factor; 8 is fast and accurate enough
YEARS    = range(2011, 2022)
PROBE    = [(2015, 6), (2019, 11)]   # Tier 2 sampled months

# REGIONS = {...}   <- use the optimized version above

mani = pd.read_csv(os.path.join(GMTDS, '_manifest.csv'))
mani.columns = [c.strip().lstrip('\ufeff') for c in mani.columns]
files = {os.path.basename(p): p for p in glob.glob(os.path.join(GMTDS, '*', 'Cargo_b*.tif'))}
print(f'tifs on disk: {len(files)}   manifest boxes: {len(mani)}')

def cells_of(boxes):
    """2° grid cell centers covered by the sub-region (unique center assignment, same rule as panel)"""
    out = set()
    for (S, N, W, E) in boxes:
        for la in np.arange(np.floor(S/GRID)*GRID, N, GRID):
            for lo in np.arange(np.floor(W/GRID)*GRID, E, GRID):
                cy, cx = la + GRID/2, lo + GRID/2
                if S <= cy < N and W <= cx < E:
                    out.add((round(cy, 3), round(cx, 3)))
    return sorted(out)

def boxes_hit(cells):
    return [r.box_id for r in mani.itertuples()
            if any(r.S <= cy <= r.N and r.W <= cx <= r.E for cy, cx in cells)]

# ---------- Tier 1: which boxes cover, are all 132 months present ----------
rows = []
for reg, subs in REGIONS.items():
    for sname, boxes in subs.items():
        cells = cells_of(boxes)
        bids  = boxes_hit(cells)
        inside = sum(any(r.S <= cy <= r.N and r.W <= cx <= r.E
                         for r in mani.itertuples() if r.box_id in bids)
                     for cy, cx in cells)
        have_month = [any(f'Cargo_b{b:02d}_{y}_{m:02d}.tif' in files for b in bids)
                      for y in YEARS for m in range(1, 13)]
        rows.append(dict(region=reg, sub=sname, cells=len(cells),
                         cells_in_footprint=inside, pct_footprint=round(inside/len(cells)*100, 1),
                         n_boxes=len(bids), boxes=','.join(map(str, bids)),
                         months=f'{sum(have_month)}/132'))
t1 = pd.DataFrame(rows)
print('\n===== Tier 1: box coverage & month completeness =====')
print(t1.to_string(index=False))
miss = t1[t1.months != '132/132']
print('\nSub-regions with missing months:', 'none' if miss.empty else miss[['sub','months']].to_string(index=False))

# ---------- Tier 2: read actual rasters ----------
def sample_month(boxes, year, month):
    cells = cells_of(boxes)
    acc   = {c: [] for c in cells}
    for bid in boxes_hit(cells):
        p = files.get(f'Cargo_b{bid:02d}_{year}_{month:02d}.tif')
        if p is None:
            continue
        with rasterio.open(p) as src:
            h, w = max(1, src.height//DECIMATE), max(1, src.width//DECIMATE)
            a = src.read(1, out_shape=(h, w)).astype('float32')
            a[a == (src.nodata if src.nodata is not None else 0)] = np.nan   # GMTDS nodata=0
            b = src.bounds
            lons = np.linspace(b.left, b.right, w, endpoint=False) + (b.right-b.left)/w/2
            lats = np.linspace(b.top, b.bottom, h, endpoint=False) - (b.top-b.bottom)/h/2
            LO, LA = np.meshgrid(lons, lats)
            ok = np.isfinite(a)
            df = pd.DataFrame({'cy': np.round(np.floor(LA/GRID)*GRID + GRID/2, 3)[ok],
                               'cx': np.round(np.floor(LO/GRID)*GRID + GRID/2, 3)[ok],
                               'v':  a[ok]})
            for (yy, xx), g in df.groupby(['cy', 'cx']):
                if (yy, xx) in acc:
                    acc[(yy, xx)].append(g.v.mean())
    return cells, {c: np.mean(v) for c, v in acc.items() if v}

rows = []
for reg, subs in REGIONS.items():
    for sname, boxes in subs.items():
        for (yy, mm) in PROBE:
            cells, vals = sample_month(boxes, yy, mm)
            rows.append(dict(region=reg, sub=sname, ym=f'{yy}-{mm:02d}',
                             cells=len(cells), with_traffic=len(vals),
                             pct=round(len(vals)/len(cells)*100, 1),
                             median_E=round(float(np.median(list(vals.values()))), 3) if vals else np.nan,
                             max_E=round(float(np.max(list(vals.values()))), 1) if vals else np.nan))
t2 = pd.DataFrame(rows)
print('\n===== Tier 2: read actual rasters =====')
print(t2.to_string(index=False))


In [ ]:
# ============================================================
# Targeted check: edge-clipped accidents vs current boundaries vs GMTDS coverage
# ============================================================
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

GMTDS_ORG = rf'{DATA_ROOT}/climate_AIS_data/GMTDS/cargo_ship/organized'
ACC       = rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet'

# --- boundaries & helpers (self-contained) ---
REGIONS = {
    "NW_Europe":   [(43, 63, -15, 0), (48, 63, 0, 10)],
    "NW_Atlantic": [(18, 50, -98, -60)],
    "SE_Asia":     [(-10, 20, 95, 127)],
    "E_Asia":      [(20, 42, 117, 145)],
    "Med_Black":   [(30, 43, -5.5, 0), (30, 47.5, 0, 42)],
    "Baltic":      [(53, 66, 10, 30)],
}
def assign_region(lat, lon):
    for r, boxes in REGIONS.items():
        for (S, N, W, E) in boxes:
            if (S <= lat < N) and (W <= lon < E): return r
    return None
def draw_region_boxes(ax, color='blue', **kw):
    st = dict(fill=False, edgecolor=color, lw=1.4); st.update(kw)
    for r, boxes in REGIONS.items():
        for (S, N, W, E) in boxes:
            ax.add_patch(plt.Rectangle((W, S), E-W, N-S, **st))

# --- accidents: take "edge-clipped" clusters (location in study region but outside box) ---
acc = pd.read_parquet(ACC).dropna(subset=['lat','lon'])
acc = acc[acc.year.between(2011,2021)].copy()
acc['region'] = [assign_region(a,o) for a,o in zip(acc.lat, acc.lon)]
out = acc[acc.region.isna()].copy()

CLIP = {
    'Baltic':                                             'Baltic',
    'Japan, Korea and North China':                       'E_Asia',
    'East Mediterranean & Black Sea':                     'Med_Black',
    'West Mediterranean':                                 'Med_Black',
    'Newfoundland':                                       'NW_Atlantic',
    'South China, Indo China, Indonesia and Philippines': 'SE_Asia',
    'Iceland and Northern Norway':                        'NW_Europe',
}
out['target'] = out['casualty_location'].map(CLIP)
clip = out[out.target.notna()].copy()
print(f'edge-clipped candidate accidents: {len(clip)}'); print(clip.groupby('target').size().to_string())

# --- GMTDS raw coverage envelope ---
mani = pd.read_csv(os.path.join(GMTDS_ORG, '_manifest.csv'))
mani.columns = [c.strip().lstrip('\ufeff') for c in mani.columns]
def in_gmtds(lat, lon):
    return any((r.S <= lat <= r.N) and (r.W <= lon <= r.E) for r in mani.itertuples())
clip['gmtds'] = [in_gmtds(a,o) for a,o in zip(clip.lat, clip.lon)]


In [ ]:
# --- per cluster: extent + whether existing GMTDS covers it (expand box or re-download) ---
print('\n=== Each edge-clipped cluster: spatial extent + GMTDS covered share ===')
for t, g in clip.groupby('target'):
    print(f'{t:12s} n={len(g):3d} | lat[{g.lat.min():5.1f},{g.lat.max():5.1f}] '
          f'lon[{g.lon.min():6.1f},{g.lon.max():6.1f}] | GMTDS covered {g.gmtds.mean()*100:3.0f}%')

In [ ]:
# ============================================================
# New bounding box visualization (with real land/coastlines)
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

REGIONS = {
    "NW_Europe":   [(43, 63, -15, 0), (48, 63, 0, 10)],
    "NW_Atlantic": [(18, 50, -98, -60)],
    "SE_Asia":     [(-10, 20, 95, 127)],
    "E_Asia":      [(20, 42, 117, 145)],
    "Med_Black":   [(30, 43, -5.5, 0), (30, 47.5, 0, 42)],
    "Baltic":      [(53, 66, 10, 30)],
}
cols = dict(zip(REGIONS, plt.cm.tab10.colors))

def add_land(ax):
    ax.add_feature(cfeature.LAND,  facecolor='#e4d9c3', zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor='#d6e6ef', zorder=0)
    ax.add_feature(cfeature.COASTLINE, lw=.5, zorder=1)
    ax.add_feature(cfeature.BORDERS,   lw=.3, ls=':', alpha=.5, zorder=1)

# ---------- Fig 1: global overview ----------
fig = plt.figure(figsize=(16, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global(); add_land(ax)
for r, boxes in REGIONS.items():
    for (S, N, W, E) in boxes:
        ax.add_patch(plt.Rectangle((W, S), E-W, N-S, transform=ccrs.PlateCarree(),
                     fill=False, edgecolor=cols[r], lw=2, zorder=3))
    S = min(b[0] for b in boxes); N = max(b[1] for b in boxes)
    W = min(b[2] for b in boxes); E = max(b[3] for b in boxes)
    ax.text((W+E)/2, N+1.5, r, color=cols[r], fontsize=9, fontweight='bold',
            ha='center', transform=ccrs.PlateCarree(), zorder=4)
gl = ax.gridlines(draw_labels=True, lw=.3, alpha=.3); gl.top_labels=gl.right_labels=False
ax.set_title('6-region bounding boxes · global overview', fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ---------- Fig 2: zoom on each of 6 regions (land + edge-clipped accidents) ----------
# if targeted check was run, overlay edge-clipped accidents from clip; else boxes only
have_clip = 'clip' in dir()

fig, axes = plt.subplots(2, 3, figsize=(19, 11),
                         subplot_kw={'projection': ccrs.PlateCarree()})
for ax, (r, boxes) in zip(axes.ravel(), REGIONS.items()):
    S = min(b[0] for b in boxes); N = max(b[1] for b in boxes)
    W = min(b[2] for b in boxes); E = max(b[3] for b in boxes)
    mg = 7
    ax.set_extent([W-mg, E+mg, S-mg, N+mg], crs=ccrs.PlateCarree())
    add_land(ax)
    for (s, n, w, e) in boxes:
        ax.add_patch(plt.Rectangle((w, s), e-w, n-s, transform=ccrs.PlateCarree(),
                     fill=False, edgecolor='red', lw=2.2, zorder=3))
    if have_clip:
        g = clip[clip.target == r]
        if len(g):
            ax.scatter(g.lon, g.lat, s=22, color='darkorange', edgecolor='k',
                       linewidth=.3, transform=ccrs.PlateCarree(), zorder=5,
                       label=f'edge-clipped accidents {len(g)}')
            ax.legend(loc='lower left', fontsize=8)
    gl = ax.gridlines(draw_labels=True, lw=.3, alpha=.3)
    gl.top_labels = gl.right_labels = False; gl.xlabel_style = gl.ylabel_style = {'size':7}
    ax.set_title(f'{r}  {boxes}', fontsize=9)
plt.suptitle('6 regions zoomed · red box = current edge · orange = edge-clipped accidents · check busy waters are not cut', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# Visualization: land + all accidents + trial westward extension of NW_Europe
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature

ACC = rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet'
REGIONS = {
    "NW_Europe":   [(43, 63, -15, 0), (48, 63, 0, 10)],
    "NW_Atlantic": [(18, 50, -98, -60)],
    "SE_Asia":     [(-10, 20, 95, 127)],
    "E_Asia":      [(20, 42, 117, 145)],
    "Med_Black":   [(30, 43, -5.5, 0), (30, 47.5, 0, 42)],
    "Baltic":      [(53, 66, 10, 30)],
}
def assign_region(lat, lon, R=REGIONS):
    for r, boxes in R.items():
        for (S,N,W,E) in boxes:
            if S<=lat<N and W<=lon<E: return r
    return None
def in_box(lat, lon, boxes):
    return any(S<=lat<N and W<=lon<E for (S,N,W,E) in boxes)

acc = pd.read_parquet(ACC).dropna(subset=['lat','lon'])
acc = acc[acc.year.between(2011,2021)].copy()
acc['region'] = [assign_region(a,o) for a,o in zip(acc.lat, acc.lon)]
out = acc[acc.region.isna()]

NWEU_WEST = -20                       # <- change here to widen (e.g. -25)
prop_nweu = [(43, 63, NWEU_WEST, 0), (48, 63, 0, 10)]
gained = out[[in_box(a,o, prop_nweu) for a,o in zip(out.lat, out.lon)]]
print(f'NW_Europe west edge -15 → {NWEU_WEST}: newly captures {len(gained)} previously outside accidents')
print(gained['casualty_location'].value_counts().head(8).to_string())


In [ ]:
def add_land(ax):
    ax.add_feature(cfeature.LAND,  facecolor='#e4d9c3', zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor='#d6e6ef', zorder=0)
    ax.add_feature(cfeature.COASTLINE, lw=.5, zorder=1)

fig, axes = plt.subplots(2, 3, figsize=(19, 11), subplot_kw={'projection': ccrs.PlateCarree()})
for ax, (r, boxes) in zip(axes.ravel(), REGIONS.items()):
    S=min(b[0] for b in boxes); N=max(b[1] for b in boxes)
    W=min(b[2] for b in boxes); E=max(b[3] for b in boxes)
    mg=7; ax.set_extent([W-mg, E+mg, S-mg, N+mg], crs=ccrs.PlateCarree()); add_land(ax)
    ext = acc[(acc.lon>W-mg)&(acc.lon<E+mg)&(acc.lat>S-mg)&(acc.lat<N+mg)]
    ki, ko = ext[ext.region==r], ext[ext.region!=r]
    ax.scatter(ko.lon, ko.lat, s=6, c='red',   alpha=.35, transform=ccrs.PlateCarree(), zorder=4, label=f'outside/other region {len(ko)}')
    ax.scatter(ki.lon, ki.lat, s=6, c='green', alpha=.55, transform=ccrs.PlateCarree(), zorder=4, label=f'this region {len(ki)}')
    for (s,n,w,e) in boxes:
        ax.add_patch(plt.Rectangle((w,s), e-w, n-s, transform=ccrs.PlateCarree(), fill=False, edgecolor='red', lw=2.2, zorder=5))
    if r == 'NW_Europe':
        for (s,n,w,e) in prop_nweu:
            ax.add_patch(plt.Rectangle((w,s), e-w, n-s, transform=ccrs.PlateCarree(),
                         fill=False, edgecolor='blue', lw=2, ls='--', zorder=6))
    gl=ax.gridlines(draw_labels=True, lw=.3, alpha=.3); gl.top_labels=gl.right_labels=False
    gl.xlabel_style=gl.ylabel_style={'size':7}
    ax.set_title(f'{r}  {boxes}', fontsize=9); ax.legend(loc='lower left', fontsize=7)
plt.suptitle('6 regions · red box = current · blue dashed = proposed NW_Europe west extension · green = in-region · red = outside/other', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ---- Europe focus: try NW_Europe southward extension + Baltic merge ----
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature

# reuse acc if still in memory; otherwise uncomment the two lines below to reload
# acc = pd.read_parquet(f'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet').dropna(subset=['lat','lon'])
# acc = acc[acc.year.between(2011,2021)].copy()

MERGE_BALTIC = False      # True -> NW_Europe absorbs Baltic
NWEU_SOUTH   = 40         # south edge; try 38 / 37 to go further (current is 43)

# current box
CUR_NWEU   = [(43, 63, -15, 0), (48, 63, 0, 15)]
CUR_BALTIC = [(53, 66, 15, 30)]
# proposed box: west edge back to -14, south edge lowered, east edge depends on Baltic merge
PROP_NWEU  = [(NWEU_SOUTH, 63, -14, -6), (43, 63, -6, 0), (48, 63, 0, 13 if MERGE_BALTIC else 10)]
if MERGE_BALTIC:
    PROP_NWEU += [(53, 66, 15, 30)]

def in_box(lat, lon, boxes): return any(S<=lat<N and W<=lon<E for (S,N,W,E) in boxes)
cur_set  = set(np.where([in_box(a,o, CUR_NWEU+CUR_BALTIC) for a,o in zip(acc.lat,acc.lon)])[0])
prop_set = set(np.where([in_box(a,o, PROP_NWEU)           for a,o in zip(acc.lat,acc.lon)])[0])
gained = acc.iloc[list(prop_set - cur_set)]
print(f'proposed (south edge {NWEU_SOUTH}, merge Baltic={MERGE_BALTIC}) vs current: net change in accidents {len(prop_set)-len(cur_set):+d}')
print('Top newly captured locations:'); print(gained['casualty_location'].value_counts().head(8).to_string())

fig = plt.figure(figsize=(13, 9)); ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-20, 34, 34, 70], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='#e4d9c3'); ax.add_feature(cfeature.OCEAN, facecolor='#d6e6ef')
ax.add_feature(cfeature.COASTLINE, lw=.5)
sub = acc[(acc.lon>-20)&(acc.lon<34)&(acc.lat>34)&(acc.lat<70)]
ax.scatter(sub.lon, sub.lat, s=7, c='#333', alpha=.4, transform=ccrs.PlateCarree(), zorder=4, label=f'accidents {len(sub)}')
for (s,n,w,e) in CUR_NWEU+CUR_BALTIC:
    ax.add_patch(plt.Rectangle((w,s),e-w,n-s,transform=ccrs.PlateCarree(),fill=False,edgecolor='red',lw=2,zorder=5))
for (s,n,w,e) in PROP_NWEU:
    ax.add_patch(plt.Rectangle((w,s),e-w,n-s,transform=ccrs.PlateCarree(),fill=False,edgecolor='blue',lw=1.8,ls='--',zorder=6))
gl=ax.gridlines(draw_labels=True,lw=.3,alpha=.3); gl.top_labels=gl.right_labels=False
ax.legend(loc='lower left'); ax.set_title(f'red = current · blue dashed = proposed (south edge {NWEU_SOUTH}, merge Baltic={MERGE_BALTIC})')
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# Targeted check: edge-clipped accidents vs current boundaries vs GMTDS coverage
# ============================================================
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

GMTDS_ORG = rf'{DATA_ROOT}/climate_AIS_data/GMTDS/cargo_ship/organized'
ACC       = rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet'

# --- boundaries & helpers (self-contained) ---
REGIONS = {
    # North Atlantic / European storm-track systems
    "NW_Europe": [
        (36, 63, -15, -6),
        (43, 63, -6, 0),
        (48, 63, 0, 13),
    ],

    "NW_Atlantic": [
        (18, 50, -98, -60),
    ],

    # Merged Asian monsoon-typhoon shipping system
    "Asia_Monsoon": [
        (-10, 20, 95, 127),   # Malacca, South China Sea, Indonesia/Philippines
        (20, 42, 117, 145),   # South China, East China Sea, Yellow/Bohai, Korea/Japan
    ],

    # Semi-enclosed / sheltered seas
    "Med_Black": [
        (30, 43, -6, 0),
        (30, 47.5, 0, 42),
    ],

    "Baltic": [
        (53, 66, 13, 30),
    ],

    # Suggested supplement: Indian Ocean monsoon-cyclone corridor
    "Arabian_BayBengal": [
        (5, 26, 50, 78),      # Arabian Sea
        (5, 26, 78, 100),     # Bay of Bengal + Andaman transition
    ],
}
def assign_region(lat, lon):
    for r, boxes in REGIONS.items():
        for (S, N, W, E) in boxes:
            if (S <= lat < N) and (W <= lon < E): return r
    return None
def draw_region_boxes(ax, color='blue', **kw):
    st = dict(fill=False, edgecolor=color, lw=1.4); st.update(kw)
    for r, boxes in REGIONS.items():
        for (S, N, W, E) in boxes:
            ax.add_patch(plt.Rectangle((W, S), E-W, N-S, **st))

# --- accidents: take "edge-clipped" clusters (location in study region but outside box) ---
acc = pd.read_parquet(ACC).dropna(subset=['lat','lon'])
acc = acc[acc.year.between(2011,2021)].copy()
acc['region'] = [assign_region(a,o) for a,o in zip(acc.lat, acc.lon)]
out = acc[acc.region.isna()].copy()

CLIP = {
    'Baltic':                                             'Baltic',
    'Japan, Korea and North China':                       'E_Asia',
    'East Mediterranean & Black Sea':                     'Med_Black',
    'West Mediterranean':                                 'Med_Black',
    'Newfoundland':                                       'NW_Atlantic',
    'South China, Indo China, Indonesia and Philippines': 'SE_Asia',
    'Iceland and Northern Norway':                        'NW_Europe',
}
out['target'] = out['casualty_location'].map(CLIP)
clip = out[out.target.notna()].copy()
print(f'edge-clipped candidate accidents: {len(clip)}'); print(clip.groupby('target').size().to_string())

# --- GMTDS raw coverage envelope ---
mani = pd.read_csv(os.path.join(GMTDS_ORG, '_manifest.csv'))
mani.columns = [c.strip().lstrip('\ufeff') for c in mani.columns]
def in_gmtds(lat, lon):
    return any((r.S <= lat <= r.N) and (r.W <= lon <= r.E) for r in mani.itertuples())
clip['gmtds'] = [in_gmtds(a,o) for a,o in zip(clip.lat, clip.lon)]


In [ ]:
# ============================================================
# New bounding box visualization (with real land/coastlines)
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# --- boundaries & helpers (self-contained) ---
REGIONS = {
    # North Atlantic / European storm-track systems
    "NW_Europe": [
        (36, 63, -15, -6),
        (43, 63, -6, 0),
        (48, 63, 0, 13),
    ],

    "NW_Atlantic": [
        (18, 50, -98, -60),
    ],

    # Merged Asian monsoon-typhoon shipping system
    "Asia_Monsoon": [
        (-10, 20, 95, 127),   # Malacca, South China Sea, Indonesia/Philippines
        (20, 42, 117, 145),   # South China, East China Sea, Yellow/Bohai, Korea/Japan
    ],

    # Semi-enclosed / sheltered seas
    "Med_Black": [
        (30, 43, -6, 0),
        (30, 47.5, 0, 42),
    ],

    "Baltic": [
        (53, 66, 13, 30),
    ],

    # Suggested supplement: Indian Ocean monsoon-cyclone corridor
    "Arabian_BayBengal": [
        (5, 26, 50, 78),      # Arabian Sea
        (5, 26, 78, 100),     # Bay of Bengal + Andaman transition
    ],
    'W_Africa': [
        (-12, 16, -20, 14)
    ]
}
cols = dict(zip(REGIONS, plt.cm.tab10.colors))

def add_land(ax):
    ax.add_feature(cfeature.LAND,  facecolor='#e4d9c3', zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor='#d6e6ef', zorder=0)
    ax.add_feature(cfeature.COASTLINE, lw=.5, zorder=1)
    ax.add_feature(cfeature.BORDERS,   lw=.3, ls=':', alpha=.5, zorder=1)

# ---------- Fig 1: global overview ----------
fig = plt.figure(figsize=(16, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global(); add_land(ax)
for r, boxes in REGIONS.items():
    for (S, N, W, E) in boxes:
        ax.add_patch(plt.Rectangle((W, S), E-W, N-S, transform=ccrs.PlateCarree(),
                     fill=False, edgecolor=cols[r], lw=2, zorder=3))
    S = min(b[0] for b in boxes); N = max(b[1] for b in boxes)
    W = min(b[2] for b in boxes); E = max(b[3] for b in boxes)
    ax.text((W+E)/2, N+1.5, r, color=cols[r], fontsize=9, fontweight='bold',
            ha='center', transform=ccrs.PlateCarree(), zorder=4)
gl = ax.gridlines(draw_labels=True, lw=.3, alpha=.3); gl.top_labels=gl.right_labels=False
ax.set_title('6-region bounding boxes · global overview', fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ---------- Fig 2: zoom on each of 6 regions (land + edge-clipped accidents) ----------
# if targeted check was run, overlay edge-clipped accidents from clip; else boxes only
have_clip = 'clip' in dir()

fig, axes = plt.subplots(2, 3, figsize=(19, 11),
                         subplot_kw={'projection': ccrs.PlateCarree()})
for ax, (r, boxes) in zip(axes.ravel(), REGIONS.items()):
    S = min(b[0] for b in boxes); N = max(b[1] for b in boxes)
    W = min(b[2] for b in boxes); E = max(b[3] for b in boxes)
    mg = 7
    ax.set_extent([W-mg, E+mg, S-mg, N+mg], crs=ccrs.PlateCarree())
    add_land(ax)
    for (s, n, w, e) in boxes:
        ax.add_patch(plt.Rectangle((w, s), e-w, n-s, transform=ccrs.PlateCarree(),
                     fill=False, edgecolor='red', lw=2.2, zorder=3))
    if have_clip:
        g = clip[clip.target == r]
        if len(g):
            ax.scatter(g.lon, g.lat, s=22, color='darkorange', edgecolor='k',
                       linewidth=.3, transform=ccrs.PlateCarree(), zorder=5,
                       label=f'edge-clipped accidents {len(g)}')
            ax.legend(loc='lower left', fontsize=8)
    gl = ax.gridlines(draw_labels=True, lw=.3, alpha=.3)
    gl.top_labels = gl.right_labels = False; gl.xlabel_style = gl.ylabel_style = {'size':7}
    ax.set_title(f'{r}  {boxes}', fontsize=9)
plt.suptitle('6 regions zoomed · red box = current edge · orange = edge-clipped accidents · check busy waters are not cut', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# Visualization: land + all accidents + trial westward extension of NW_Europe
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature

ACC = rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet'
REGIONS = {
    "NW_Europe":   [(43, 63, -15, 0), (48, 63, 0, 10)],
    "NW_Atlantic": [(18, 50, -98, -60)],
    "SE_Asia":     [(-10, 20, 95, 127)],
    "E_Asia":      [(20, 42, 117, 145)],
    "Med_Black":   [(30, 43, -5.5, 0), (30, 47.5, 0, 42)],
    "Baltic":      [(53, 66, 10, 30)],
}
def assign_region(lat, lon, R=REGIONS):
    for r, boxes in R.items():
        for (S,N,W,E) in boxes:
            if S<=lat<N and W<=lon<E: return r
    return None
def in_box(lat, lon, boxes):
    return any(S<=lat<N and W<=lon<E for (S,N,W,E) in boxes)

acc = pd.read_parquet(ACC).dropna(subset=['lat','lon'])
acc = acc[acc.year.between(2011,2021)].copy()
acc['region'] = [assign_region(a,o) for a,o in zip(acc.lat, acc.lon)]
out = acc[acc.region.isna()]

NWEU_WEST = -20                       # <- change here to widen (e.g. -25)
prop_nweu = [(43, 63, NWEU_WEST, 0), (48, 63, 0, 10)]
gained = out[[in_box(a,o, prop_nweu) for a,o in zip(out.lat, out.lon)]]
print(f'NW_Europe west edge -15 → {NWEU_WEST}: newly captures {len(gained)} previously outside accidents')
print(gained['casualty_location'].value_counts().head(8).to_string())


In [ ]:
def add_land(ax):
    ax.add_feature(cfeature.LAND,  facecolor='#e4d9c3', zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor='#d6e6ef', zorder=0)
    ax.add_feature(cfeature.COASTLINE, lw=.5, zorder=1)

fig, axes = plt.subplots(2, 3, figsize=(19, 11), subplot_kw={'projection': ccrs.PlateCarree()})
for ax, (r, boxes) in zip(axes.ravel(), REGIONS.items()):
    S=min(b[0] for b in boxes); N=max(b[1] for b in boxes)
    W=min(b[2] for b in boxes); E=max(b[3] for b in boxes)
    mg=7; ax.set_extent([W-mg, E+mg, S-mg, N+mg], crs=ccrs.PlateCarree()); add_land(ax)
    ext = acc[(acc.lon>W-mg)&(acc.lon<E+mg)&(acc.lat>S-mg)&(acc.lat<N+mg)]
    ki, ko = ext[ext.region==r], ext[ext.region!=r]
    ax.scatter(ko.lon, ko.lat, s=6, c='red',   alpha=.35, transform=ccrs.PlateCarree(), zorder=4, label=f'outside/other region {len(ko)}')
    ax.scatter(ki.lon, ki.lat, s=6, c='green', alpha=.55, transform=ccrs.PlateCarree(), zorder=4, label=f'this region {len(ki)}')
    for (s,n,w,e) in boxes:
        ax.add_patch(plt.Rectangle((w,s), e-w, n-s, transform=ccrs.PlateCarree(), fill=False, edgecolor='red', lw=2.2, zorder=5))
    if r == 'NW_Europe':
        for (s,n,w,e) in prop_nweu:
            ax.add_patch(plt.Rectangle((w,s), e-w, n-s, transform=ccrs.PlateCarree(),
                         fill=False, edgecolor='blue', lw=2, ls='--', zorder=6))
    gl=ax.gridlines(draw_labels=True, lw=.3, alpha=.3); gl.top_labels=gl.right_labels=False
    gl.xlabel_style=gl.ylabel_style={'size':7}
    ax.set_title(f'{r}  {boxes}', fontsize=9); ax.legend(loc='lower left', fontsize=7)
plt.suptitle('6 regions · red box = current · blue dashed = proposed NW_Europe west extension · green = in-region · red = outside/other', y=1.01)
plt.tight_layout(); plt.show()
